In [1]:
siamese_accuracies = []
siamese_precisions = []
siamese_recalls = []
siamese_f1s = []
siamese_roc_aucs = []
siamese_balanced_accuracies = []
siamese_tnrs = []

siamese_imbalanced_accuracies = []
siamese_imbalanced_precisions = []
siamese_imbalanced_recalls = []
siamese_imbalanced_f1s = []
siamese_imbalanced_roc_aucs = []
siamese_imbalanced_balanced_accuracies = []
siamese_imbalanced_tnrs = []

diplomski_accuracies = []
diplomski_precisions = []
diplomski_recalls = []
diplomski_f1s = []

ExoplANNET_accuracies = []
ExoplANNET_precisions = []
ExoplANNET_recalls = []
ExoplANNET_f1s = []

# Diplomski extended metrics (balanced)
diplomski_roc_aucs = []
diplomski_balanced_accuracies = []
diplomski_tnrs = []

# Diplomski imbalanced metrics
diplomski_imbalanced_accuracies = []
diplomski_imbalanced_precisions = []
diplomski_imbalanced_recalls = []
diplomski_imbalanced_f1s = []
diplomski_imbalanced_roc_aucs = []
diplomski_imbalanced_balanced_accuracies = []
diplomski_imbalanced_tnrs = []

# ExoplANNET extended metrics (balanced)
ExoplANNET_roc_aucs = []
ExoplANNET_balanced_accuracies = []
ExoplANNET_tnrs = []

# ExoplANNET imbalanced metrics
ExoplANNET_imbalanced_accuracies = []
ExoplANNET_imbalanced_precisions = []
ExoplANNET_imbalanced_recalls = []
ExoplANNET_imbalanced_f1s = []
ExoplANNET_imbalanced_roc_aucs = []
ExoplANNET_imbalanced_balanced_accuracies = []
ExoplANNET_imbalanced_tnrs = []

# Accumulators for precision-recall curve data across seeds (all models)
all_y_true_balanced = []
all_y_scores_balanced = []
all_y_true_imbalanced = []
all_y_scores_imbalanced = []

all_y_true_diplomski_balanced = []
all_y_scores_diplomski_balanced = []
all_y_true_diplomski_imbalanced = []
all_y_scores_diplomski_imbalanced = []

all_y_true_ExoplANNET_balanced = []
all_y_scores_ExoplANNET_balanced = []
all_y_true_ExoplANNET_imbalanced = []
all_y_scores_ExoplANNET_imbalanced = []

In [2]:
for seed in range(42, 47):
    def seed_everything(seed):
        import random
        import torch
        import numpy as np
    
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        
    seed_everything(seed)



    import pandas as pd
    import numpy as np
    import re
    from astropy.io import fits
    import matplotlib.pyplot as plt

    
    
    def parse_image(image_str):
        image_str = image_str.strip('[] ')
        
        rows = re.split(r'\]\s*\[', image_str)
        array = []
        for row in rows:
            elements = row.split()
            array.append([float(e) for e in elements])
        return np.array(array)
    
    # harps_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/data.csv', converters={'image': parse_image}))
    harps_images = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-images/HARPS_data_with_candidates_and_phase.pkl'))
    # harps_images = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-images/HARPS_data_with_candidates_and_phase_random_shift.pkl'))
    # harps_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/HARPS_GAFs_with_Candidates.csv', converters = {'image': parse_image}))
    # hires_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/HIRES_GAFs.csv', converters={'image': parse_image}))
    hires_images = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-images/HIRES_data_with_candidates_and_phase.pkl'))
    # hires_images = pd.DataFrame(pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-images/HIRES_data_with_candidates_and_phase_random_shift.pkl'))
    # hires_images = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/harps-images/HIRES_GAFs_with_candidates.csv', converters = {'image': parse_image}))
    hires_df = pd.read_csv('/kaggle/input/datasets/maanav0114/harps-n-dataset/preprocessed_HIRES_data.csv')
    with fits.open('/kaggle/input/datasets/maanav0114/harps-n-dataset/ADP.2023-12-04T15_16_53.464.fits') as data:
        # Convert to native byte order immediately
            fits_data = data[1].data
            
            # Create DataFrame with native byte order arrays
            df_dict = {}
            for name in fits_data.dtype.names:
                arr = fits_data[name]
                # Convert to native byte order
                if not arr.dtype.isnative:
                    arr = arr.astype(arr.dtype.newbyteorder('='))
                df_dict[name] = arr
            
            harps_df = pd.DataFrame(df_dict)
    
    
    
    
    
    from torchvision import transforms
    
    
    
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    
    
    
    
    
    import torch
    from torch.utils.data import DataLoader, Dataset
    from sklearn.model_selection import train_test_split
    import random
    
    
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    
    positives = harps_images[harps_images['label'] == 1].sample(n=(len(harps_images[harps_images['label'] == 1])))
    negatives = harps_images[harps_images['label'] == 0].sample(n=(len(harps_images[harps_images['label'] == 1])))
    
    train_pos, temp_pos = train_test_split(positives, test_size=0.3, random_state=seed)
    val_pos, test_pos = train_test_split(temp_pos, test_size=0.5, random_state=seed)  
    
    train_neg, temp_neg = train_test_split(negatives, test_size=0.3, random_state=seed)
    val_neg, test_neg = train_test_split(temp_neg, test_size=0.5, random_state=seed)  
    
    
    
    hires_positives = hires_images[hires_images['label'] == 1].sample(n=(len(hires_images[hires_images['label'] == 1])))
    hires_negatives = hires_images[hires_images['label'] == 0].sample(n=(len(hires_images[hires_images['label'] == 1])))
    
    hires_train_pos, hires_temp_pos = train_test_split(hires_positives, test_size=0.3, random_state=seed)
    hires_val_pos, hires_test_pos = train_test_split(hires_temp_pos, test_size=0.5, random_state=seed)  
    
    hires_train_neg, hires_temp_neg = train_test_split(hires_negatives, test_size=0.3, random_state=seed)
    hires_val_neg, hires_test_neg = train_test_split(hires_temp_neg, test_size=0.5, random_state=seed)
    
    
    
    train_data = np.array((pd.concat([train_pos, train_neg, hires_train_pos, hires_train_neg]))['image'])
    val_data = np.array((pd.concat([val_pos, val_neg, hires_val_pos, hires_val_neg]))['image'])
    test_data = pd.concat([test_pos, test_neg, hires_test_pos, hires_test_neg])
    
    
    
    class TripletDataset(Dataset):
        def __init__(self, triplets):
            self.triplets = triplets
    
        def __len__(self):
            return len(self.triplets)
    
        def __getitem__(self, idx):
            anchor, pos, neg = self.triplets[idx]
            
            return anchor, pos, neg
    
    
    
    
    class ImageDataset(Dataset):
        def __init__(self, images, labels):
            self.data = images
            self.labels = labels
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            image = torch.from_numpy(self.data[idx])
            label = torch.tensor(self.labels[idx])
    
            return image, label
    
    
    train_labels = []
    val_labels = []
    test_labels = []
    
    for i in range(len(train_pos) + len(hires_train_pos)):
        train_labels.append(1)
    for i in range(len(train_neg) + len(hires_train_neg)):
        train_labels.append(0)
    
    for i in range(len(val_pos) + len(hires_val_pos)):
        val_labels.append(1)
    for i in range(len(val_neg) + len(hires_val_neg)):
        val_labels.append(0)
    
    for i in range(len(test_pos) + len(hires_test_pos)):
        test_labels.append(1)
    for i in range(len(test_neg) + len(hires_test_neg)):
        test_labels.append(0)
    
    train_dataset = ImageDataset(train_data, train_labels)
    val_dataset = ImageDataset(val_data, val_labels)
    # test_dataset = ImageDataset(test_data, test_labels)
    
    
    batch_size = 32
    
    train_data = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        # batch_size = len(train_dataset),
        shuffle=True,
        drop_last=False,
        pin_memory=True if device.type == 'cuda' else False 
    )
    
    val_data = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        # batch_size = len(val_dataset),
        shuffle=False,
        drop_last=False,
        pin_memory=True if device.type == 'cuda' else False
    )


    test_pos = pd.concat([test_pos, hires_test_pos])
    test_neg = pd.concat([test_neg, hires_test_neg])
    
    # print(f'Data splits created. Train size: {len(train_data) * batch_size} Val Size: {len(val_data) * batch_size} Test Size: {len(test_data)}')
    print(f'Data splits created. Train size: {len(train_data) * batch_size} Val Size: {len(val_data) * batch_size} Test Size: {len(test_data)}')
    
    
    
    
    import torch.nn as nn
    import torch.nn.functional as F
    
    
    
    # class HARPSNet(nn.Module):
    #     def __init__(self):
    #         super().__init__()
    #         self.encoder = nn.Sequential(
    #             # Block 1: Input 1x128x128 -> Output 16x64x64
    #             nn.Conv2d(1, 16, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(16),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
    
    #             # Block 2: Input 16x64x64 -> Output 32x32x32
    #             nn.Conv2d(16, 32, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(32),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
    
    #             # Block 3: Input 32x32x32 -> Output 64x16x16
    #             nn.Conv2d(32, 64, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(64),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
                
    #             # Block 4: Input 64x16x16 -> Output 128x8x8
    #             nn.Conv2d(64, 128, kernel_size=3, padding=1),
    #             nn.BatchNorm2d(128),
    #             nn.ReLU(inplace=True),
    #             nn.MaxPool2d(2, 2),
    
    #             nn.Flatten(),
                
    #             # The feature map is now 128 channels * 8 * 8 spatial size = 8192
    #             # This is the same size as before, but it contains much richer, 
    #             # high-level semantic info rather than low-level noise.
    #             nn.Linear(128 * 8 * 8, 512),
    #             nn.ReLU(inplace=True),
    #             # nn.Dropout(0.5), # Increased dropout for the larger layer 
                
    #             # Optional: Add a second linear layer if you have many classes
    #             # nn.Linear(512, 256) 
    #             # Note: If this is for classification, you'd usually add your final 
    #             # class projection here (e.g., nn.Linear(256, num_classes))
    #         )
    
    #     def forward(self, x):
    #         x = self.encoder(x)
    #         x = F.normalize(x, p = 2, dim = 1)
    #         return x

    #         # return self.encoder(x)




    class SelfAttentionBlock(nn.Module):
        def __init__(self, in_channels, spatial_size):
            super().__init__()
            self.norm = nn.LayerNorm(in_channels)
            # Suggestion: Use more than 1 head (e.g., 4 or 8) to capture different relationships
            self.mha = nn.MultiheadAttention(embed_dim=in_channels, num_heads=8, batch_first=True)
            
            # Learnable Positional Encoding: Sequence Length (H*W) x Channels
            self.pos_embedding = nn.Parameter(torch.randn(1, spatial_size * spatial_size, in_channels))
            
            # SAGAN style gating
            self.scale = nn.Parameter(torch.zeros(1))
    
        def forward(self, x):
            bs, c, h, w = x.shape
            
            # Reshape to (Batch, Seq_Len, Dim)
            x_flat = x.reshape(bs, c, h * w).transpose(1, 2)
            
            # Apply Norm
            x_norm = self.norm(x_flat)
            
            # Add Positional Embedding (Broadcasting across batch)
            x_norm = x_norm + self.pos_embedding
            
            # Apply Attention
            # key_padding_mask is not needed here as images are fixed size
            attn_out, _ = self.mha(x_norm, x_norm, x_norm)
            
            # Reshape back to (Batch, Dim, H, W)
            attn_out = attn_out.transpose(1, 2).reshape(bs, c, h, w)
            
            # Residual connection with Scale
            return self.scale * attn_out + x

    class HARPSNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                # Block 1: 1 -> 16
                nn.Conv2d(1, 16, kernel_size=3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
        
                # Block 2: 16 -> 32
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
        
                # Block 3: 32 -> 64
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
                
                # Block 4: 64 -> 128 (Output size 8x8)
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )
    
            # Initialize Attention Block
            # We know the output is 128 channels and size 8x8
            self.attention = SelfAttentionBlock(in_channels=128, spatial_size=8)
    
            self.flatten = nn.Flatten()
            self.linear = nn.Linear(128 * 8 * 8, 512)
            self.relu = nn.ReLU(inplace=True)
            self.do = nn.Dropout(0.5)
            # Add final classification layer if needed
            # self.final = nn.Linear(512, num_classes)
    
        def forward(self, x):
            x = self.encoder(x)
            
            # Apply Attention
            x = self.attention(x)
            
            x = self.flatten(x)
            x = self.linear(x)
            x = self.relu(x)
            x = self.do(x)
            
            return x
    
    
    
    
    model = HARPSNet()
    
    
    
    
    
    !pip install -q pytorch-metric-learning
    
    
    
    
    import torch.optim
    from pytorch_metric_learning import losses, miners
    from torch.optim.lr_scheduler import ReduceLROnPlateau
    
    
    
    model = model.to(device)
    
    
    
    # optimizer = torch.optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
    # optimizer = torch.optim.SGD(model.parameters(), lr=0.00001, momentum=0.9, weight_decay = 0.0001)
    # optimizer = torch.optim.Adam(model.parameters(), lr = 0.00001, weight_decay = 0.0001)
    # optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0001, weight_decay = 0.01)
    
    # criterion = nn.BCELoss()
    # criterion = ContrastiveLoss()
    margin = 0.7
    criterion = nn.TripletMarginLoss(margin = margin, p = 2)
    
    # margin = 14.32394
    # margin = 1
    # criterion = losses.ArcFaceLoss(num_classes = 2, embedding_size = 256, margin = margin, scale = 64)
    optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001, weight_decay = 0.1)
    
    
    
    miner = miners.TripletMarginMiner(
        margin=margin,
        type_of_triplets="semihard"
    )
    
    # scheduler = ReduceLROnPlateau(
    #     optimizer,
    #     mode='min',         
                            
    #     factor=0.5,         
    #     patience=10,         
    #     min_lr=0.001,     
    #     verbose=True        
    # )
    
    
    
    train_losses = []
    val_losses = [100]
    epochs = 500
    patience = 10
    patience_counter = 0
    
    for epoch in range(epochs):
        epoch_loss = 0
        model.train()
        
        for images, labels in train_data:
            images = images.float().to(device)
            labels = labels.to(device)
    
            optimizer.zero_grad()
    
            current_embeddings = model.forward(images.view(-1, 1, 128, 128))
            hard_triplet_indices = miner(current_embeddings, labels)
            if hard_triplet_indices[0].shape[0] == 0:
                # print(f"Epoch {epoch}: No semi-hard triplets found in this batch. Skipping update.")
                continue
    
            emb_anchors = current_embeddings[hard_triplet_indices[0]]
            emb_positives = current_embeddings[hard_triplet_indices[1]]
            emb_negatives = current_embeddings[hard_triplet_indices[2]]
            
            loss = criterion(emb_anchors, emb_positives, emb_negatives)
            # loss = criterion(current_embeddings, labels)
    
            loss.backward()
            optimizer.step()
    
            epoch_loss += loss.item()
    
        avg_train_loss = epoch_loss / len(train_data)
        train_losses.append(avg_train_loss)
    
    
        epoch_loss = 0
        model.eval()
        with torch.no_grad():
            for images, labels in val_data:
                images = images.float().to(device)
                labels = labels.to(device)
    
                current_embeddings = model.forward(images.view(-1, 1, 128, 128))
                hard_triplet_indices = miner(current_embeddings, labels)
                if hard_triplet_indices[0].shape[0] == 0:
                    # print(f"Epoch {epoch}: No semi-hard triplets found in this batch. Skipping.")
                    continue # Skip this batch if no triplets can be formed
        
                emb_anchors = current_embeddings[hard_triplet_indices[0]]
                emb_positives = current_embeddings[hard_triplet_indices[1]]
                emb_negatives = current_embeddings[hard_triplet_indices[2]]
            
                loss = criterion(emb_anchors, emb_positives, emb_negatives)
                # loss = criterion(current_embeddings, labels)
        
                epoch_loss += loss.item()
                
        avg_val_loss = epoch_loss / len(val_data)
        val_losses.append(avg_val_loss)
        
    
        if epoch % 100 == 0:
            print(f'Epoch: {epoch} | Average Train Loss: {avg_train_loss}\t Average Validation Loss: {avg_val_loss}')
    
    
        if avg_val_loss < val_losses[-2]:
            patience_counter = 0
        else:
            patience_counter += 1
    
    
        if avg_val_loss == sorted(val_losses)[0]:
            torch.save(model.state_dict(), '/kaggle/working/model.pth')
            # print(f'model saved at: \t{avg_val_loss}')
    
        
        if patience_counter >= patience:
            print("Patience reached")
            break
    
    
    val_losses.pop(0)
    print("Model training finished")

    plt.figure(figsize=(16,9))
    plt.plot(train_losses, c='b', label='Train loss')
    plt.plot(val_losses, c='r', label = 'Validation loss')
    plt.legend()
    plt.grid()
    plt.xlabel('Epochs', fontsize=20)
    plt.ylabel('Loss', fontsize=20)
    plt.title(f'Losses for seed: {seed}')
    
    
    
    
    
    import h5py as h5
    from astropy.timeseries import LombScargle
    
    
    
    def gen_periodogram(star_name):
            try:
                # Filter and clean data
                filtered = harps_df[harps_df['main_id_simbad'] == star_name]
                filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
                
                # Skip if insufficient data
                if len(filtered) < 3:
                    # Filter and clean data
                    filtered = hires_df[hires_df['main_id_simbad'] == star_name]
                    filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
                    
                    # Skip if insufficient data
                    if len(filtered) < 3:
                        return 1, 1  # Placeholder for "not enough data"
                    
                    # Extract arrays with explicit native conversion
                    def to_native_float64(series):
                        arr = np.array(series)
                        if not arr.dtype.isnative:
                            arr = arr.astype(arr.dtype.newbyteorder('='))
                        return arr.astype(np.float64)
                    
                    time = to_native_float64(filtered['drs_bjd'])
                    rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
                    uncertainty = to_native_float64(filtered['drs_dvrms'])
                    
                    # Normalize timestamps for numerical stability
                    time -= np.min(time)
                    
                    # Generate periodogram
                    periodogram = LombScargle(time, rad_vel, uncertainty)
                    frequency, power = periodogram.autopower()
                    
                    # Resample to 1000 points
                    freq_uniform = np.linspace(frequency.min(), frequency.max(), 1000)
                    power_interp = np.interp(freq_uniform, frequency, power)
                    
                    return freq_uniform, power_interp
                    
                
                # Extract arrays with explicit native conversion
                def to_native_float64(series):
                    arr = np.array(series)
                    if not arr.dtype.isnative:
                        arr = arr.astype(arr.dtype.newbyteorder('='))
                    return arr.astype(np.float64)
                
                time = to_native_float64(filtered['drs_bjd'])
                rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
                uncertainty = to_native_float64(filtered['drs_dvrms'])
                
                # Normalize timestamps for numerical stability
                time -= np.min(time)
                
                # Generate periodogram
                periodogram = LombScargle(time, rad_vel, uncertainty)
                frequency, power = periodogram.autopower()
                
                # Resample to 1000 points
                freq_uniform = np.linspace(frequency.min(), frequency.max(), 1000)
                power_interp = np.interp(freq_uniform, frequency, power)
                
                return freq_uniform, power_interp
                
            except Exception as e:
                print(f"Error processing {star_name}: {e}")
                return 1, 1
    
    
    
    stars = list(test_data['star'])
    
    filename = "/kaggle/working/processed_data.h5"
    with h5.File(filename, 'w') as file:
            for star in stars:
                frequency, power = gen_periodogram(star)
                if type(frequency) == int:
                    pass
                else:
                    # star_group = file.create_group(clean_name(star))
                    star_group = file.create_group(star)
                    star_group.create_dataset('frequencies', data = frequency)
                    star_group.create_dataset('power', data = power)
    
    print("Test data h5 file created for Ante's model.")
    #Evaluate on test data using Ante's Model
    !python3 /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/modified_validateRealData.py /kaggle/working/processed_data.h5 /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/top_current_model_trained_on_uneven_data_fully.pth --threshold 0.73 --catalog_path /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/catalog_of_exoplanets.csv
    
    
    preds = pd.DataFrame(pd.read_csv('/kaggle/working/planet_predictions.csv'))
    star_level = pd.DataFrame(columns = preds.columns)
    # test_data = pd.DataFrame(pd.read_csv('/kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/test_data.csv')).drop('Unnamed: 0', axis = 1)
    stars = list(set(preds['Star Name']))
    preds['Is True Planet'] = preds['Is True Planet'].astype(int)
    
    for i in range(len(stars)):
        star_level.loc[i, 'Star Name'] = stars[i]
    
    # Collect raw continuous scores per star BEFORE thresholding
    diplomski_raw_scores = {}
    for star in stars:
        star_scores = list(preds[preds['Star Name'] == star]['Prediction'])
        diplomski_raw_scores[star] = max(star_scores)
    
    for i in range(len(preds)):
        if preds.iloc[i]['Prediction'] > 0.73:
            preds.loc[i, 'Prediction'] = 1.0
        else:
            preds.loc[i, 'Prediction'] = 0.0
    
        
        # if preds.iloc[i]['Is True Planet'] == True:
        #     preds.loc[i, 'Is True Planet'] = 1.0
        # else:
        #     preds.loc[i, 'Is True Planet'] = 0.0
    
    
    for i in range(len(stars)):
        star = stars[i]
        prediction = min(sum(list(preds[preds['Star Name'] == star]['Prediction'])), 1)
        star_level.loc[i, 'Prediction'] = prediction
    
        # label = sum(list(preds[preds['Star Name'] == star]['Is True Planet']))
        try:
            # label = test_data[test_data['star'] == star].iloc[0][-1]
            for idx in range(len(test_data)):
                if star in test_data.iloc[idx][0]:
                    label = test_data.iloc[idx][-1]
                    break
        except Exception as e:
            print(star)
            print(test_data[test_data['star'] == star])
        # print(label)
        try:
            star_level.loc[i, 'Is True Planet'] = label
        except Exception as e:
            print(e)
            print(test_data.iloc[idx][0])
    
    
    
    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0
    
    for i in range(len(star_level)):
        if star_level.iloc[i]['Prediction'] == star_level.iloc[i]['Is True Planet']:
            if star_level.iloc[i]['Is True Planet'] == 0.0:
                true_neg += 1
            elif star_level.iloc[i]['Is True Planet'] == 1.0:
                true_pos += 1
        else:
            if star_level.iloc[i]['Is True Planet'] == 0.0:
                false_pos += 1
            elif star_level.iloc[i]['Is True Planet'] == 1.0:
                false_neg += 1
    
    
    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg)
    precision = true_pos / (true_pos + false_pos)
    recall = true_pos / (true_pos + false_neg)
    f1 = (2 * precision * recall) / (precision + recall)
    
    diplomski_accuracies.append(accuracy)
    diplomski_precisions.append(precision)
    diplomski_recalls.append(recall)
    diplomski_f1s.append(f1)
    
    # Extended metrics for Diplomski (balanced)
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    tnr_d_bal = true_neg / (true_neg + false_pos) if (true_neg + false_pos) > 0 else 0
    diplomski_tnrs.append(tnr_d_bal)
    
    diplomski_y_true_bal = []
    diplomski_y_scores_bal = []
    for i in range(len(star_level)):
        star_name_d = star_level.iloc[i]['Star Name']
        diplomski_y_true_bal.append(int(float(star_level.iloc[i]['Is True Planet'])))
        diplomski_y_scores_bal.append(diplomski_raw_scores.get(star_name_d, 0.0))
    
    try:
        diplomski_roc_auc_bal = roc_auc_score(diplomski_y_true_bal, diplomski_y_scores_bal)
    except:
        diplomski_roc_auc_bal = 0.5
    diplomski_y_pred_bal = [1 if s > 0.73 else 0 for s in diplomski_y_scores_bal]
    diplomski_balanced_acc_bal = balanced_accuracy_score(diplomski_y_true_bal, diplomski_y_pred_bal)
    
    diplomski_roc_aucs.append(diplomski_roc_auc_bal)
    diplomski_balanced_accuracies.append(diplomski_balanced_acc_bal)
    all_y_true_diplomski_balanced.append(diplomski_y_true_bal)
    all_y_scores_diplomski_balanced.append(diplomski_y_scores_bal)
    
    # Save balanced confusion matrix for imbalanced evaluation later
    diplomski_bal_tp = true_pos
    diplomski_bal_tn = true_neg
    diplomski_bal_fp = false_pos
    diplomski_bal_fn = false_neg
    
    
    
    
    
    preds = pd.DataFrame(pd.read_csv('/kaggle/working/planet_predictions.csv'))
    pred_stars = list(set(preds['Star Name']))
    
    # test_df = pd.read_csv('/kaggle/working/test_data.csv', converters={'image': parse_image}).drop('Unnamed: 0', axis = 1)
    temp = pd.DataFrame(columns = test_data.columns)
    stars = list(test_data['star'])
    for pred_star in pred_stars:
        # print("".join(pred_star.split()))
        for star in stars:
            if "".join(pred_star.split()) == "".join(star.split()):
            # score = fuzz.ratio(pred_star, star)
            # if score >= 91:
                temp = pd.concat([temp, test_data[test_data['star'] == star]])
                break
    
    test_pos = temp[temp['label'] == 1]
    test_neg = temp[temp['label'] == 0]
    
    
    

    del model
    model = HARPSNet().to(device)
    model.load_state_dict(torch.load('/kaggle/working/model.pth'))
    model.eval()
    
    pos_embeddings = []
    
    with torch.no_grad():
        for i in range(len(train_pos)):
            image = torch.tensor(train_pos.iloc[i]['image']).float().to(device).view(1, -1, 128, 128)
            embedding = model.forward(image)
            pos_embeddings.append(embedding.detach())
    
    stacked_embeddings = torch.cat(pos_embeddings, dim=0)
    mean_vector = torch.mean(stacked_embeddings, dim=0)
    pos_prototype = F.normalize(mean_vector, p=2, dim=0).view(1, -1)
    
    
    neg_embeddings = []
    
    with torch.no_grad():
        for i in range(len(train_neg)):
            image = torch.tensor(train_neg.iloc[i]['image']).float().to(device).view(1, -1, 128, 128)
            embedding = model.forward(image)
            neg_embeddings.append(embedding.detach())
    
    stacked_embeddings = torch.cat(neg_embeddings, dim=0)
    mean_vector = torch.mean(stacked_embeddings, dim=0)
    neg_prototype = F.normalize(mean_vector, p=2, dim=0).view(1, -1)





    # from sklearn.neighbors import KNeighborsClassifier



    # train_embeddings = []
    # train_labels_np = []

    # with torch.no_grad():
    #     # assuming train_data is your DataLoader
    #     for images, labels in train_data:
    #         images = images.float().to(device).view(-1, 1, 128, 128) # Adjust size as needed
    #         emb = model.encoder(images)
    #         emb = emb.view(emb.size(0), -1) # Flatten the embeddings
    #         emb = F.normalize(emb, p=2, dim=1)
    #         train_embeddings.append(emb.cpu().numpy())
    #         train_labels_np.extend(labels.cpu().numpy())

    # train_embeddings = np.vstack(train_embeddings)
    # train_labels_np = np.array(train_labels_np)

    # # 2. Fit k-NN Classifier
    # knn = KNeighborsClassifier(n_neighbors=3, metric='cosine') #nga
    # knn.fit(train_embeddings, train_labels_np)





    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0

    y_true = []
    y_scores = []

    for idx in range(len(test_pos)):
        test_image = torch.tensor(test_pos.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

        with torch.no_grad():
            embedding = model.forward(test_image)

        pos_sim = F.cosine_similarity(embedding, pos_prototype)
        neg_sim = F.cosine_similarity(embedding, neg_prototype)
        score = (pos_sim - neg_sim).item()
        
        y_true.append(1)
        y_scores.append(score)

        if pos_sim > neg_sim:
            true_pos += 1
        else:
            false_neg += 1

    for idx in range(len(test_neg)):
        test_image = torch.tensor(test_neg.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

        with torch.no_grad():
            embedding = model.forward(test_image)

        pos_sim = F.cosine_similarity(embedding, pos_prototype)
        neg_sim = F.cosine_similarity(embedding, neg_prototype)
        score = (pos_sim - neg_sim).item()

        y_true.append(0)
        y_scores.append(score)

        if pos_sim > neg_sim:
            false_pos += 1
        else:
            true_neg += 1



    # test_embeddings = []
    # test_labels_real = []

    # for idx in range(len(test_pos)):
    #     test_image = torch.tensor(test_pos.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

    #     with torch.no_grad():
    #         emb = model.encoder(test_image)
    #         emb = emb.view(emb.size(0), -1) # Flatten the embeddings

    #     test_embeddings.append(emb.cpu().numpy())
    #     test_labels_real.append(1)


    # for idx in range(len(test_neg)):
    #     test_image = torch.tensor(test_neg.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

    #     with torch.no_grad():
    #         emb = model.encoder(test_image)
    #         emb = emb.view(emb.size(0), -1) # Flatten the embeddings

    #     test_embeddings.append(emb.cpu().numpy())
    #     test_labels_real.append(0)

    # test_embeddings = np.vstack(test_embeddings)

    # predictions = knn.predict(test_embeddings)

    # for idx in range(len(test_labels_real)):
    #     label = test_labels_real[idx]
    #     pred = predictions[idx]

    #     if label == 1:
    #         if pred == 1:
    #             true_pos += 1
    #         else:
    #             false_neg += 1
    #     else:
    #         if pred == 0:
    #             true_neg += 1
    #         else:
    #             false_pos += 1

    
    
    
    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg) if (true_pos + true_neg + false_pos + false_neg) > 0 else 0
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    tnr = true_neg / (true_neg + false_pos) if (true_neg + false_pos) > 0 else 0
    
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    try:
        roc_auc = roc_auc_score(y_true, y_scores)
    except:
        roc_auc = 0.5
    y_pred = [1 if s > 0 else 0 for s in y_scores]
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    
    siamese_accuracies.append(accuracy)
    siamese_precisions.append(precision)
    siamese_recalls.append(recall)
    siamese_f1s.append(f1)
    siamese_roc_aucs.append(roc_auc)
    siamese_balanced_accuracies.append(balanced_acc)
    siamese_tnrs.append(tnr)

    # Save per-seed PR curve data (balanced)
    all_y_true_balanced.append(y_true)
    all_y_scores_balanced.append(y_scores)



    # Evaluating on imbalanced dataset (extra negatives)
    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0

    unused_negatives = harps_images[harps_images['label'] == 0]
    unused_negatives = unused_negatives[~unused_negatives['image'].isin(test_neg['image'])]

    test_neg_imbalanced = pd.concat([test_neg, unused_negatives])

    y_true_imb = []
    y_scores_imb = []


    for idx in range(len(test_pos)):
        test_image = torch.tensor(test_pos.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

        with torch.no_grad():
            embedding = model.forward(test_image)

        pos_sim = F.cosine_similarity(embedding, pos_prototype)
        neg_sim = F.cosine_similarity(embedding, neg_prototype)
        score = (pos_sim - neg_sim).item()
        
        y_true_imb.append(1)
        y_scores_imb.append(score)

        if pos_sim > neg_sim:
            true_pos += 1
        else:
            false_neg += 1

    for idx in range(len(test_neg_imbalanced)):
        test_image = torch.tensor(test_neg_imbalanced.iloc[idx]['image']).float().to(device).view(1, -1, 128, 128)

        with torch.no_grad():
            embedding = model.forward(test_image)

        pos_sim = F.cosine_similarity(embedding, pos_prototype)
        neg_sim = F.cosine_similarity(embedding, neg_prototype)
        score = (pos_sim - neg_sim).item()
        
        y_true_imb.append(0)
        y_scores_imb.append(score)

        if pos_sim > neg_sim:
            false_pos += 1
        else:
            true_neg += 1

    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg) if (true_pos + true_neg + false_pos + false_neg) > 0 else 0
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    tnr = true_neg / (true_neg + false_pos) if (true_neg + false_pos) > 0 else 0
    try:
        roc_auc_imb = roc_auc_score(y_true_imb, y_scores_imb)
    except:
        roc_auc_imb = 0.5
    y_pred_imb = [1 if s > 0 else 0 for s in y_scores_imb]
    balanced_acc_imb = balanced_accuracy_score(y_true_imb, y_pred_imb)
    
    siamese_imbalanced_accuracies.append(accuracy)
    siamese_imbalanced_precisions.append(precision)
    siamese_imbalanced_recalls.append(recall)
    siamese_imbalanced_f1s.append(f1)
    siamese_imbalanced_roc_aucs.append(roc_auc_imb)
    siamese_imbalanced_balanced_accuracies.append(balanced_acc_imb)
    siamese_imbalanced_tnrs.append(tnr)

    # Save per-seed PR curve data (imbalanced)
    all_y_true_imbalanced.append(y_true_imb)
    all_y_scores_imbalanced.append(y_scores_imb)
    
    print("Model Evaluation Complete")
    
    
    
    
    
    def gen_pg(star_name):
        try:
            # Filter and clean data
            filtered = harps_df[harps_df['main_id_simbad'] == star_name]
            filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
            
            # Skip if insufficient data
            if len(filtered) < 3:
                filtered = hires_df[hires_df['main_id_simbad'] == star_name]
                filtered = filtered.dropna(subset=['drs_bjd', 'drs_ccf_rvc', 'drs_dvrms'])
                
                # Skip if insufficient data
                if len(filtered) < 3:
                    return 1, 1  # Placeholder for "not enough data"
                
                # Extract arrays with explicit native conversion
                def to_native_float64(series):
                    arr = np.array(series)
                    if not arr.dtype.isnative:
                        arr = arr.astype(arr.dtype.newbyteorder('='))
                    return arr.astype(np.float64)
                
                time = to_native_float64(filtered['drs_bjd'])
                rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
                uncertainty = to_native_float64(filtered['drs_dvrms'])
                
                # Normalize timestamps for numerical stability
                time -= np.min(time)
                
                # Generate periodogram
                periodogram = LombScargle(time, rad_vel, uncertainty)
                frequency, power = periodogram.autopower()
                
                # Resample to 1000 points
                freq_uniform = np.linspace(frequency.min(), frequency.max(), 990)
                power_interp = np.interp(freq_uniform, frequency, power)
                
                return freq_uniform, power_interp
            
            # Extract arrays with explicit native conversion
            def to_native_float64(series):
                arr = np.array(series)
                if not arr.dtype.isnative:
                    arr = arr.astype(arr.dtype.newbyteorder('='))
                return arr.astype(np.float64)
            
            time = to_native_float64(filtered['drs_bjd'])
            rad_vel = to_native_float64(filtered['drs_ccf_rvc'])
            uncertainty = to_native_float64(filtered['drs_dvrms'])
            
            # Normalize timestamps for numerical stability
            time -= np.min(time)
            
            # Generate periodogram
            periodogram = LombScargle(time, rad_vel, uncertainty)
            frequency, power = periodogram.autopower()
            
            # Resample to 1000 points
            freq_uniform = np.linspace(frequency.min(), frequency.max(), 990)
            power_interp = np.interp(freq_uniform, frequency, power)
            
            return freq_uniform, power_interp
            
        except Exception as e:
            print(f"Error processing {star_name}: {e}")
            return 1, 1
    
    
    def gen_peak_pow(power):
        peak_pow = max(list(power))
        peak = list(power).index(max(list(power)))
    
        return peak, peak_pow
    
    
    
    
    
    
    from tensorflow import keras
    from keras.models import Model
    # from keras.utils.vis_utils import plot_model
    from keras.utils import plot_model #I had to make this change, as the import statement above is not supported by current keras version
    from keras.models import load_model
    import tensorflow as tf
    
    
    
    model = load_model("/kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/exoplANNET_trained.h5")
    
    def run_inference(star_name):
        freq, power = gen_pg(star_name)
        peak, peak_pow = gen_peak_pow(power)
    
        pg = np.transpose(np.array([power]))
        peak_pow = np.transpose(np.array([[peak, peak_pow]]))
    
        pg = np.expand_dims(pg, 0)
        peak_pow = np.expand_dims(peak_pow, 0)
    
        output = model.predict([pg, peak_pow])
        # print(output)
        if output < 0.77:
            output = 0
        else:
            output = 1
    
        return output
    
    def run_inference_with_score(star_name):
        """Returns (binary_prediction, raw_score). Returns (None, None) on error."""
        try:
            freq, power = gen_pg(star_name)
            if type(freq) == int:
                return None, None
            peak, peak_pow_val = gen_peak_pow(power)
            pg = np.transpose(np.array([power]))
            peak_pow_arr = np.transpose(np.array([[peak, peak_pow_val]]))
            pg = np.expand_dims(pg, 0)
            peak_pow_arr = np.expand_dims(peak_pow_arr, 0)
            raw_output = float(model.predict([pg, peak_pow_arr]))
            binary_output = 1 if raw_output >= 0.77 else 0
            return binary_output, raw_output
        except Exception as e:
            return None, None
    
    
    
    predictions = {}
    exoplannet_raw_scores = {}
    for star in pred_stars:
        binary_pred, raw_score = run_inference_with_score(star)
        if binary_pred is not None:
            predictions[star] = binary_pred
            exoplannet_raw_scores[star] = raw_score
        else:
            output = run_inference(star)
            predictions[star] = output
            exoplannet_raw_scores[star] = 0.0
    
    
    
    
    
    true_pos = 0
    true_neg = 0
    false_pos = 0
    false_neg = 0
    
    exo_y_true_bal = []
    exo_y_scores_bal = []
    
    for star in predictions:
        prediction = predictions[star]
        label = int(test_data[test_data['star'] == star]['label'])
    
        exo_y_true_bal.append(label)
        exo_y_scores_bal.append(exoplannet_raw_scores[star])
    
        if label == 1:
            if prediction == 1:
                true_pos += 1
            else:
                false_neg += 1
        else:
            if prediction == 1:
                false_pos += 1
            else:
                true_neg += 1
    
    
    accuracy = (true_pos + true_neg) / (true_pos + true_neg + false_pos + false_neg)
    precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0
    recall = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    ExoplANNET_accuracies.append(accuracy)
    ExoplANNET_precisions.append(precision)
    ExoplANNET_recalls.append(recall)
    ExoplANNET_f1s.append(f1)
    
    # Extended metrics for ExoplANNET (balanced)
    tnr_e_bal = true_neg / (true_neg + false_pos) if (true_neg + false_pos) > 0 else 0
    ExoplANNET_tnrs.append(tnr_e_bal)
    
    try:
        exo_roc_auc_bal = roc_auc_score(exo_y_true_bal, exo_y_scores_bal)
    except:
        exo_roc_auc_bal = 0.5
    exo_y_pred_bal = [1 if s >= 0.77 else 0 for s in exo_y_scores_bal]
    exo_balanced_acc_bal = balanced_accuracy_score(exo_y_true_bal, exo_y_pred_bal)
    
    ExoplANNET_roc_aucs.append(exo_roc_auc_bal)
    ExoplANNET_balanced_accuracies.append(exo_balanced_acc_bal)
    all_y_true_ExoplANNET_balanced.append(exo_y_true_bal)
    all_y_scores_ExoplANNET_balanced.append(exo_y_scores_bal)
    
    # Save balanced confusion matrix for imbalanced reuse
    exo_bal_tp = true_pos
    exo_bal_tn = true_neg
    exo_bal_fp = false_pos
    exo_bal_fn = false_neg
    
    print('ExoplANNET evaluation complete.')
    
    # ============================================================
    # BASELINE IMBALANCED EVALUATIONS
    # ============================================================
    
    # Define extra negative stars (same set as Siamese imbalanced test)
    all_harps_neg_star_names = set(harps_images[harps_images['label'] == 0]['star'])
    balanced_test_star_names = set(test_data['star'])
    extra_neg_star_names = list(all_harps_neg_star_names - balanced_test_star_names)
    print(f"Extra negative stars for imbalanced baseline evaluation: {len(extra_neg_star_names)}")
    
    # --- Diplomski Imbalanced Evaluation ---
    # Generate periodograms for extra negative stars
    filename_imb = "/kaggle/working/processed_data_imbalanced.h5"
    with h5.File(filename_imb, 'w') as file:
        for star in extra_neg_star_names:
            frequency, power = gen_periodogram(star)
            if type(frequency) != int:
                try:
                    star_group = file.create_group(star)
                    star_group.create_dataset('frequencies', data=frequency)
                    star_group.create_dataset('power', data=power)
                except:
                    pass
    
    # Run Diplomski script on extra negatives h5 file
    !python3 /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/modified_validateRealData.py /kaggle/working/processed_data_imbalanced.h5 /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/top_current_model_trained_on_uneven_data_fully.pth --threshold 0.73 --catalog_path /kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/catalog_of_exoplanets.csv
    
    # Read extra negative predictions
    try:
        preds_imb_extra = pd.DataFrame(pd.read_csv('/kaggle/working/planet_predictions.csv'))
        extra_pred_stars = list(set(preds_imb_extra['Star Name']))
        
        # Collect raw scores for extra negative stars
        extra_diplomski_raw = {}
        for star in extra_pred_stars:
            star_scores = list(preds_imb_extra[preds_imb_extra['Star Name'] == star]['Prediction'])
            extra_diplomski_raw[star] = max(star_scores)
        
        # Threshold extra predictions for binary evaluation
        extra_diplomski_binary = {}
        for i in range(len(preds_imb_extra)):
            if preds_imb_extra.iloc[i]['Prediction'] > 0.73:
                preds_imb_extra.loc[i, 'Prediction'] = 1.0
            else:
                preds_imb_extra.loc[i, 'Prediction'] = 0.0
        for star in extra_pred_stars:
            extra_diplomski_binary[star] = min(sum(list(preds_imb_extra[preds_imb_extra['Star Name'] == star]['Prediction'])), 1)
    except:
        extra_pred_stars = []
        extra_diplomski_raw = {}
        extra_diplomski_binary = {}
    
    # Combine balanced + extra for imbalanced evaluation
    d_imb_tp = diplomski_bal_tp
    d_imb_tn = diplomski_bal_tn
    d_imb_fp = diplomski_bal_fp
    d_imb_fn = diplomski_bal_fn
    
    diplomski_y_true_imb = list(diplomski_y_true_bal)
    diplomski_y_scores_imb = list(diplomski_y_scores_bal)
    
    for star in extra_pred_stars:
        raw_score = extra_diplomski_raw.get(star, 0.0)
        binary_pred = extra_diplomski_binary.get(star, 0.0)
        diplomski_y_true_imb.append(0)  # All extra stars are negatives
        diplomski_y_scores_imb.append(raw_score)
        if binary_pred == 1.0:
            d_imb_fp += 1
        else:
            d_imb_tn += 1
    
    # Stars that couldn't generate periodograms => default negative prediction
    processed_extra = set(extra_pred_stars)
    for star in extra_neg_star_names:
        if star not in processed_extra:
            diplomski_y_true_imb.append(0)
            diplomski_y_scores_imb.append(0.0)
            d_imb_tn += 1
    
    d_imb_accuracy = (d_imb_tp + d_imb_tn) / (d_imb_tp + d_imb_tn + d_imb_fp + d_imb_fn) if (d_imb_tp + d_imb_tn + d_imb_fp + d_imb_fn) > 0 else 0
    d_imb_precision = d_imb_tp / (d_imb_tp + d_imb_fp) if (d_imb_tp + d_imb_fp) > 0 else 0
    d_imb_recall = d_imb_tp / (d_imb_tp + d_imb_fn) if (d_imb_tp + d_imb_fn) > 0 else 0
    d_imb_f1 = (2 * d_imb_precision * d_imb_recall) / (d_imb_precision + d_imb_recall) if (d_imb_precision + d_imb_recall) > 0 else 0
    d_imb_tnr = d_imb_tn / (d_imb_tn + d_imb_fp) if (d_imb_tn + d_imb_fp) > 0 else 0
    
    try:
        d_imb_roc_auc = roc_auc_score(diplomski_y_true_imb, diplomski_y_scores_imb)
    except:
        d_imb_roc_auc = 0.5
    d_imb_y_pred = [1 if s > 0.73 else 0 for s in diplomski_y_scores_imb]
    d_imb_balanced_acc = balanced_accuracy_score(diplomski_y_true_imb, d_imb_y_pred)
    
    diplomski_imbalanced_accuracies.append(d_imb_accuracy)
    diplomski_imbalanced_precisions.append(d_imb_precision)
    diplomski_imbalanced_recalls.append(d_imb_recall)
    diplomski_imbalanced_f1s.append(d_imb_f1)
    diplomski_imbalanced_roc_aucs.append(d_imb_roc_auc)
    diplomski_imbalanced_balanced_accuracies.append(d_imb_balanced_acc)
    diplomski_imbalanced_tnrs.append(d_imb_tnr)
    all_y_true_diplomski_imbalanced.append(diplomski_y_true_imb)
    all_y_scores_diplomski_imbalanced.append(diplomski_y_scores_imb)
    
    print('Diplomski imbalanced evaluation complete.')
    
    # --- ExoplANNET Imbalanced Evaluation ---
    e_imb_tp = exo_bal_tp
    e_imb_tn = exo_bal_tn
    e_imb_fp = exo_bal_fp
    e_imb_fn = exo_bal_fn
    
    exo_y_true_imb = list(exo_y_true_bal)
    exo_y_scores_imb = list(exo_y_scores_bal)
    
    for star in extra_neg_star_names:
        binary_pred, raw_score = run_inference_with_score(star)
        if binary_pred is None:
            # Can't process this star, default to negative
            exo_y_true_imb.append(0)
            exo_y_scores_imb.append(0.0)
            e_imb_tn += 1
        else:
            exo_y_true_imb.append(0)  # All extra are negatives
            exo_y_scores_imb.append(raw_score)
            if binary_pred == 1:
                e_imb_fp += 1
            else:
                e_imb_tn += 1
    
    e_imb_accuracy = (e_imb_tp + e_imb_tn) / (e_imb_tp + e_imb_tn + e_imb_fp + e_imb_fn) if (e_imb_tp + e_imb_tn + e_imb_fp + e_imb_fn) > 0 else 0
    e_imb_precision = e_imb_tp / (e_imb_tp + e_imb_fp) if (e_imb_tp + e_imb_fp) > 0 else 0
    e_imb_recall = e_imb_tp / (e_imb_tp + e_imb_fn) if (e_imb_tp + e_imb_fn) > 0 else 0
    e_imb_f1 = (2 * e_imb_precision * e_imb_recall) / (e_imb_precision + e_imb_recall) if (e_imb_precision + e_imb_recall) > 0 else 0
    e_imb_tnr = e_imb_tn / (e_imb_tn + e_imb_fp) if (e_imb_tn + e_imb_fp) > 0 else 0
    
    try:
        e_imb_roc_auc = roc_auc_score(exo_y_true_imb, exo_y_scores_imb)
    except:
        e_imb_roc_auc = 0.5
    e_imb_y_pred = [1 if s >= 0.77 else 0 for s in exo_y_scores_imb]
    e_imb_balanced_acc = balanced_accuracy_score(exo_y_true_imb, e_imb_y_pred)
    
    ExoplANNET_imbalanced_accuracies.append(e_imb_accuracy)
    ExoplANNET_imbalanced_precisions.append(e_imb_precision)
    ExoplANNET_imbalanced_recalls.append(e_imb_recall)
    ExoplANNET_imbalanced_f1s.append(e_imb_f1)
    ExoplANNET_imbalanced_roc_aucs.append(e_imb_roc_auc)
    ExoplANNET_imbalanced_balanced_accuracies.append(e_imb_balanced_acc)
    ExoplANNET_imbalanced_tnrs.append(e_imb_tnr)
    all_y_true_ExoplANNET_imbalanced.append(exo_y_true_imb)
    all_y_scores_ExoplANNET_imbalanced.append(exo_y_scores_imb)
    
    print('ExoplANNET imbalanced evaluation complete.')
    print('All evaluations complete for seed:', seed)
    # break

In [3]:
def std_dev(metric):
    mean = sum(metric)/len(metric)

    sq_diffs = 0
    for data in metric:
        diff = (data - mean) ** 2
        sq_diffs += diff
    sq_diffs = sq_diffs / len(metric)

    return sq_diffs ** 0.5

In [4]:
siamese_mean_accuracy = sum(siamese_accuracies)/len(siamese_accuracies)
siamese_mean_precision = sum(siamese_precisions)/len(siamese_precisions)
siamese_mean_recall = sum(siamese_recalls)/len(siamese_recalls)
siamese_mean_f1 = sum(siamese_f1s)/len(siamese_f1s)
siamese_mean_roc_auc = sum(siamese_roc_aucs)/len(siamese_roc_aucs)
siamese_mean_balanced_accuracy = sum(siamese_balanced_accuracies)/len(siamese_balanced_accuracies)
siamese_mean_tnr = sum(siamese_tnrs)/len(siamese_tnrs)

print(f'siamese_mean_accuracy: {siamese_mean_accuracy}\tsiamese_accuracy_std_dev: {std_dev(siamese_accuracies)}')
print(f'siamese_mean_precision: {siamese_mean_precision}\tsiamese_precision_std_dev: {std_dev(siamese_precisions)}')
print(f'siamese_mean_recall: {siamese_mean_recall}\tsiamese_recall_std_dev: {std_dev(siamese_recalls)}')
print(f'siamese_mean_f1: {siamese_mean_f1}\tsiamese_f1_std_dev: {std_dev(siamese_f1s)}')
print(f'siamese_mean_roc_auc: {siamese_mean_roc_auc}\tsiamese_roc_auc_std_dev: {std_dev(siamese_roc_aucs)}')
print(f'siamese_mean_balanced_accuracy: {siamese_mean_balanced_accuracy}\tsiamese_balanced_accuracy_std_dev: {std_dev(siamese_balanced_accuracies)}')
print(f'siamese_mean_tnr: {siamese_mean_tnr}\tsiamese_tnr_std_dev: {std_dev(siamese_tnrs)}')

In [5]:
siamese_mean_imbalanced_accuracy = sum(siamese_imbalanced_accuracies)/len(siamese_imbalanced_accuracies)
siamese_mean_imbalanced_precision = sum(siamese_imbalanced_precisions)/len(siamese_imbalanced_precisions)
siamese_mean_imbalanced_recall = sum(siamese_imbalanced_recalls)/len(siamese_imbalanced_recalls)
siamese_mean_imbalanced_f1 = sum(siamese_imbalanced_f1s)/len(siamese_imbalanced_f1s)
siamese_mean_imbalanced_roc_auc = sum(siamese_imbalanced_roc_aucs)/len(siamese_imbalanced_roc_aucs)
siamese_mean_imbalanced_balanced_accuracy = sum(siamese_imbalanced_balanced_accuracies)/len(siamese_imbalanced_balanced_accuracies)
siamese_mean_imbalanced_tnr = sum(siamese_imbalanced_tnrs)/len(siamese_imbalanced_tnrs)

print(f'siamese_mean_imbalanced_accuracy: {siamese_mean_imbalanced_accuracy}\tsiamese_imbalanced_accuracy_std_dev: {std_dev(siamese_imbalanced_accuracies)}')
print(f'siamese_mean_imbalanced_precision: {siamese_mean_imbalanced_precision}\tsiamese_imbalanced_precision_std_dev: {std_dev(siamese_imbalanced_precisions)}')
print(f'siamese_mean_imbalanced_recall: {siamese_mean_imbalanced_recall}\tsiamese_imbalanced_recall_std_dev: {std_dev(siamese_imbalanced_recalls)}')
print(f'siamese_mean_imbalanced_f1: {siamese_mean_imbalanced_f1}\tsiamese_imbalanced_f1_std_dev: {std_dev(siamese_imbalanced_f1s)}')
print(f'siamese_mean_imbalanced_roc_auc: {siamese_mean_imbalanced_roc_auc}\tsiamese_imbalanced_roc_auc_std_dev: {std_dev(siamese_imbalanced_roc_aucs)}')
print(f'siamese_mean_imbalanced_balanced_accuracy: {siamese_mean_imbalanced_balanced_accuracy}\tsiamese_imbalanced_balanced_accuracy_std_dev: {std_dev(siamese_imbalanced_balanced_accuracies)}')
print(f'siamese_mean_imbalanced_tnr: {siamese_mean_imbalanced_tnr}\tsiamese_imbalanced_tnr_std_dev: {std_dev(siamese_imbalanced_tnrs)}')

In [6]:
diplomski_mean_accuracy = sum(diplomski_accuracies)/len(diplomski_accuracies)
diplomski_mean_precision = sum(diplomski_precisions)/len(diplomski_precisions)
diplomski_mean_recall = sum(diplomski_recalls)/len(diplomski_recalls)
diplomski_mean_f1 = sum(diplomski_f1s)/len(diplomski_f1s)
diplomski_mean_roc_auc = sum(diplomski_roc_aucs)/len(diplomski_roc_aucs)
diplomski_mean_balanced_accuracy = sum(diplomski_balanced_accuracies)/len(diplomski_balanced_accuracies)
diplomski_mean_tnr = sum(diplomski_tnrs)/len(diplomski_tnrs)

print(f'diplomski_mean_accuracy: {diplomski_mean_accuracy}\tdiplomski_accuracy_std_dev: {std_dev(diplomski_accuracies)}')
print(f'diplomski_mean_precision: {diplomski_mean_precision}\tdiplomski_precision_std_dev: {std_dev(diplomski_precisions)}')
print(f'diplomski_mean_recall: {diplomski_mean_recall}\tdiplomski_recall_std_dev: {std_dev(diplomski_recalls)}')
print(f'diplomski_mean_f1: {diplomski_mean_f1}\tdiplomski_f1_std_dev: {std_dev(diplomski_f1s)}')
print(f'diplomski_mean_roc_auc: {diplomski_mean_roc_auc}\tdiplomski_roc_auc_std_dev: {std_dev(diplomski_roc_aucs)}')
print(f'diplomski_mean_balanced_accuracy: {diplomski_mean_balanced_accuracy}\tdiplomski_balanced_accuracy_std_dev: {std_dev(diplomski_balanced_accuracies)}')
print(f'diplomski_mean_tnr: {diplomski_mean_tnr}\tdiplomski_tnr_std_dev: {std_dev(diplomski_tnrs)}')

In [7]:
diplomski_mean_imbalanced_accuracy = sum(diplomski_imbalanced_accuracies)/len(diplomski_imbalanced_accuracies)
diplomski_mean_imbalanced_precision = sum(diplomski_imbalanced_precisions)/len(diplomski_imbalanced_precisions)
diplomski_mean_imbalanced_recall = sum(diplomski_imbalanced_recalls)/len(diplomski_imbalanced_recalls)
diplomski_mean_imbalanced_f1 = sum(diplomski_imbalanced_f1s)/len(diplomski_imbalanced_f1s)
diplomski_mean_imbalanced_roc_auc = sum(diplomski_imbalanced_roc_aucs)/len(diplomski_imbalanced_roc_aucs)
diplomski_mean_imbalanced_balanced_accuracy = sum(diplomski_imbalanced_balanced_accuracies)/len(diplomski_imbalanced_balanced_accuracies)
diplomski_mean_imbalanced_tnr = sum(diplomski_imbalanced_tnrs)/len(diplomski_imbalanced_tnrs)

print(f'diplomski_mean_imbalanced_accuracy: {diplomski_mean_imbalanced_accuracy}\tdiplomski_imbalanced_accuracy_std_dev: {std_dev(diplomski_imbalanced_accuracies)}')
print(f'diplomski_mean_imbalanced_precision: {diplomski_mean_imbalanced_precision}\tdiplomski_imbalanced_precision_std_dev: {std_dev(diplomski_imbalanced_precisions)}')
print(f'diplomski_mean_imbalanced_recall: {diplomski_mean_imbalanced_recall}\tdiplomski_imbalanced_recall_std_dev: {std_dev(diplomski_imbalanced_recalls)}')
print(f'diplomski_mean_imbalanced_f1: {diplomski_mean_imbalanced_f1}\tdiplomski_imbalanced_f1_std_dev: {std_dev(diplomski_imbalanced_f1s)}')
print(f'diplomski_mean_imbalanced_roc_auc: {diplomski_mean_imbalanced_roc_auc}\tdiplomski_imbalanced_roc_auc_std_dev: {std_dev(diplomski_imbalanced_roc_aucs)}')
print(f'diplomski_mean_imbalanced_balanced_accuracy: {diplomski_mean_imbalanced_balanced_accuracy}\tdiplomski_imbalanced_balanced_accuracy_std_dev: {std_dev(diplomski_imbalanced_balanced_accuracies)}')
print(f'diplomski_mean_imbalanced_tnr: {diplomski_mean_imbalanced_tnr}\tdiplomski_imbalanced_tnr_std_dev: {std_dev(diplomski_imbalanced_tnrs)}')

In [ ]:
ExoplANNET_mean_accuracy = sum(ExoplANNET_accuracies)/len(ExoplANNET_accuracies)
ExoplANNET_mean_precision = sum(ExoplANNET_precisions)/len(ExoplANNET_precisions)
ExoplANNET_mean_recall = sum(ExoplANNET_recalls)/len(ExoplANNET_recalls)
ExoplANNET_mean_f1 = sum(ExoplANNET_f1s)/len(ExoplANNET_f1s)
ExoplANNET_mean_roc_auc = sum(ExoplANNET_roc_aucs)/len(ExoplANNET_roc_aucs)
ExoplANNET_mean_balanced_accuracy = sum(ExoplANNET_balanced_accuracies)/len(ExoplANNET_balanced_accuracies)
ExoplANNET_mean_tnr = sum(ExoplANNET_tnrs)/len(ExoplANNET_tnrs)

print(f'ExoplANNET_mean_accuracy: {ExoplANNET_mean_accuracy}\tExoplANNET_accuracy_std_dev: {std_dev(ExoplANNET_accuracies)}')
print(f'ExoplANNET_mean_precision: {ExoplANNET_mean_precision}\tExoplANNET_precision_std_dev: {std_dev(ExoplANNET_precisions)}')
print(f'ExoplANNET_mean_recall: {ExoplANNET_mean_recall}\tExoplANNET_recall_std_dev: {std_dev(ExoplANNET_recalls)}')
print(f'ExoplANNET_mean_f1: {ExoplANNET_mean_f1}\tExoplANNET_f1_std_dev: {std_dev(ExoplANNET_f1s)}')
print(f'ExoplANNET_mean_roc_auc: {ExoplANNET_mean_roc_auc}\tExoplANNET_roc_auc_std_dev: {std_dev(ExoplANNET_roc_aucs)}')
print(f'ExoplANNET_mean_balanced_accuracy: {ExoplANNET_mean_balanced_accuracy}\tExoplANNET_balanced_accuracy_std_dev: {std_dev(ExoplANNET_balanced_accuracies)}')
print(f'ExoplANNET_mean_tnr: {ExoplANNET_mean_tnr}\tExoplANNET_tnr_std_dev: {std_dev(ExoplANNET_tnrs)}')

In [ ]:
ExoplANNET_mean_imbalanced_accuracy = sum(ExoplANNET_imbalanced_accuracies)/len(ExoplANNET_imbalanced_accuracies)
ExoplANNET_mean_imbalanced_precision = sum(ExoplANNET_imbalanced_precisions)/len(ExoplANNET_imbalanced_precisions)
ExoplANNET_mean_imbalanced_recall = sum(ExoplANNET_imbalanced_recalls)/len(ExoplANNET_imbalanced_recalls)
ExoplANNET_mean_imbalanced_f1 = sum(ExoplANNET_imbalanced_f1s)/len(ExoplANNET_imbalanced_f1s)
ExoplANNET_mean_imbalanced_roc_auc = sum(ExoplANNET_imbalanced_roc_aucs)/len(ExoplANNET_imbalanced_roc_aucs)
ExoplANNET_mean_imbalanced_balanced_accuracy = sum(ExoplANNET_imbalanced_balanced_accuracies)/len(ExoplANNET_imbalanced_balanced_accuracies)
ExoplANNET_mean_imbalanced_tnr = sum(ExoplANNET_imbalanced_tnrs)/len(ExoplANNET_imbalanced_tnrs)

print(f'ExoplANNET_mean_imbalanced_accuracy: {ExoplANNET_mean_imbalanced_accuracy}\tExoplANNET_imbalanced_accuracy_std_dev: {std_dev(ExoplANNET_imbalanced_accuracies)}')
print(f'ExoplANNET_mean_imbalanced_precision: {ExoplANNET_mean_imbalanced_precision}\tExoplANNET_imbalanced_precision_std_dev: {std_dev(ExoplANNET_imbalanced_precisions)}')
print(f'ExoplANNET_mean_imbalanced_recall: {ExoplANNET_mean_imbalanced_recall}\tExoplANNET_imbalanced_recall_std_dev: {std_dev(ExoplANNET_imbalanced_recalls)}')
print(f'ExoplANNET_mean_imbalanced_f1: {ExoplANNET_mean_imbalanced_f1}\tExoplANNET_imbalanced_f1_std_dev: {std_dev(ExoplANNET_imbalanced_f1s)}')
print(f'ExoplANNET_mean_imbalanced_roc_auc: {ExoplANNET_mean_imbalanced_roc_auc}\tExoplANNET_imbalanced_roc_auc_std_dev: {std_dev(ExoplANNET_imbalanced_roc_aucs)}')
print(f'ExoplANNET_mean_imbalanced_balanced_accuracy: {ExoplANNET_mean_imbalanced_balanced_accuracy}\tExoplANNET_imbalanced_balanced_accuracy_std_dev: {std_dev(ExoplANNET_imbalanced_balanced_accuracies)}')
print(f'ExoplANNET_mean_imbalanced_tnr: {ExoplANNET_mean_imbalanced_tnr}\tExoplANNET_imbalanced_tnr_std_dev: {std_dev(ExoplANNET_imbalanced_tnrs)}')

## Precision-Recall Curves: All Models — Balanced vs Imbalanced Test Sets

In [8]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score

# Helper to compute mean PR curve across seeds
def mean_pr_curve(all_y_true_list, all_y_scores_list):
    """Compute mean precision at common recall points across seeds."""
    common_recall = np.linspace(0, 1, 200)
    precisions_interp = []
    for i in range(len(all_y_true_list)):
        y_t = np.array(all_y_true_list[i])
        y_s = np.array(all_y_scores_list[i])
        p, r, _ = precision_recall_curve(y_t, y_s)
        # Interpolate precision at common recall points (reversed for monotonicity)
        prec_interp = np.interp(common_recall, r[::-1], p[::-1])
        precisions_interp.append(prec_interp)
    mean_prec = np.mean(precisions_interp, axis=0)
    std_prec = np.std(precisions_interp, axis=0)
    return common_recall, mean_prec, std_prec

def compute_mean_ap(all_y_true_list, all_y_scores_list):
    aps = []
    for i in range(len(all_y_true_list)):
        y_t = np.array(all_y_true_list[i])
        y_s = np.array(all_y_scores_list[i])
        aps.append(average_precision_score(y_t, y_s))
    return np.mean(aps), np.std(aps)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Color scheme for models
colors = {'Siamese': '#2196F3', 'Diplomski': '#FF9800', 'ExoplANNET': '#4CAF50'}

# --- Left plot: Balanced test set ---
ax = axes[0]

# Siamese
recall_pts, mean_prec, std_prec = mean_pr_curve(all_y_true_balanced, all_y_scores_balanced)
ap_mean, ap_std = compute_mean_ap(all_y_true_balanced, all_y_scores_balanced)
ax.plot(recall_pts, mean_prec, color=colors['Siamese'], linewidth=2,
        label=f'Siamese (AP={ap_mean:.3f}±{ap_std:.3f})')
ax.fill_between(recall_pts, mean_prec - std_prec, mean_prec + std_prec,
                color=colors['Siamese'], alpha=0.15)

# Diplomski
if len(all_y_true_diplomski_balanced) > 0:
    recall_pts, mean_prec, std_prec = mean_pr_curve(all_y_true_diplomski_balanced, all_y_scores_diplomski_balanced)
    ap_mean, ap_std = compute_mean_ap(all_y_true_diplomski_balanced, all_y_scores_diplomski_balanced)
    ax.plot(recall_pts, mean_prec, color=colors['Diplomski'], linewidth=2,
            label=f'Diplomski (AP={ap_mean:.3f}±{ap_std:.3f})')
    ax.fill_between(recall_pts, mean_prec - std_prec, mean_prec + std_prec,
                    color=colors['Diplomski'], alpha=0.15)

# ExoplANNET
if len(all_y_true_ExoplANNET_balanced) > 0:
    recall_pts, mean_prec, std_prec = mean_pr_curve(all_y_true_ExoplANNET_balanced, all_y_scores_ExoplANNET_balanced)
    ap_mean, ap_std = compute_mean_ap(all_y_true_ExoplANNET_balanced, all_y_scores_ExoplANNET_balanced)
    ax.plot(recall_pts, mean_prec, color=colors['ExoplANNET'], linewidth=2,
            label=f'ExoplANNET (AP={ap_mean:.3f}±{ap_std:.3f})')
    ax.fill_between(recall_pts, mean_prec - std_prec, mean_prec + std_prec,
                    color=colors['ExoplANNET'], alpha=0.15)

ax.set_xlabel('Recall', fontsize=14)
ax.set_ylabel('Precision', fontsize=14)
ax.set_title('Precision-Recall Curve — Balanced Test Set', fontsize=16)
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

# --- Right plot: Imbalanced test set ---
ax = axes[1]

# Siamese
recall_pts, mean_prec, std_prec = mean_pr_curve(all_y_true_imbalanced, all_y_scores_imbalanced)
ap_mean, ap_std = compute_mean_ap(all_y_true_imbalanced, all_y_scores_imbalanced)
ax.plot(recall_pts, mean_prec, color=colors['Siamese'], linewidth=2,
        label=f'Siamese (AP={ap_mean:.3f}±{ap_std:.3f})')
ax.fill_between(recall_pts, mean_prec - std_prec, mean_prec + std_prec,
                color=colors['Siamese'], alpha=0.15)

# Diplomski
if len(all_y_true_diplomski_imbalanced) > 0:
    recall_pts, mean_prec, std_prec = mean_pr_curve(all_y_true_diplomski_imbalanced, all_y_scores_diplomski_imbalanced)
    ap_mean, ap_std = compute_mean_ap(all_y_true_diplomski_imbalanced, all_y_scores_diplomski_imbalanced)
    ax.plot(recall_pts, mean_prec, color=colors['Diplomski'], linewidth=2,
            label=f'Diplomski (AP={ap_mean:.3f}±{ap_std:.3f})')
    ax.fill_between(recall_pts, mean_prec - std_prec, mean_prec + std_prec,
                    color=colors['Diplomski'], alpha=0.15)

# ExoplANNET
if len(all_y_true_ExoplANNET_imbalanced) > 0:
    recall_pts, mean_prec, std_prec = mean_pr_curve(all_y_true_ExoplANNET_imbalanced, all_y_scores_ExoplANNET_imbalanced)
    ap_mean, ap_std = compute_mean_ap(all_y_true_ExoplANNET_imbalanced, all_y_scores_ExoplANNET_imbalanced)
    ax.plot(recall_pts, mean_prec, color=colors['ExoplANNET'], linewidth=2,
            label=f'ExoplANNET (AP={ap_mean:.3f}±{ap_std:.3f})')
    ax.fill_between(recall_pts, mean_prec - std_prec, mean_prec + std_prec,
                    color=colors['ExoplANNET'], alpha=0.15)

ax.set_xlabel('Recall', fontsize=14)
ax.set_ylabel('Precision', fontsize=14)
ax.set_title('Precision-Recall Curve — Imbalanced Test Set', fontsize=16)
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('precision_recall_curves_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print comprehensive AP summary table
print("="*80)
print("Average Precision (AP) Summary — All Models")
print("="*80)
print(f"{'Model':<15} {'Balanced AP':>15} {'Imbalanced AP':>15}")
print("-"*50)

ap_bal_s, _ = compute_mean_ap(all_y_true_balanced, all_y_scores_balanced)
ap_imb_s, _ = compute_mean_ap(all_y_true_imbalanced, all_y_scores_imbalanced)
print(f"{'Siamese':<15} {ap_bal_s:>15.4f} {ap_imb_s:>15.4f}")

if len(all_y_true_diplomski_balanced) > 0:
    ap_bal_d, _ = compute_mean_ap(all_y_true_diplomski_balanced, all_y_scores_diplomski_balanced)
    ap_imb_d, _ = compute_mean_ap(all_y_true_diplomski_imbalanced, all_y_scores_diplomski_imbalanced)
    print(f"{'Diplomski':<15} {ap_bal_d:>15.4f} {ap_imb_d:>15.4f}")

if len(all_y_true_ExoplANNET_balanced) > 0:
    ap_bal_e, _ = compute_mean_ap(all_y_true_ExoplANNET_balanced, all_y_scores_ExoplANNET_balanced)
    ap_imb_e, _ = compute_mean_ap(all_y_true_ExoplANNET_imbalanced, all_y_scores_ExoplANNET_imbalanced)
    print(f"{'ExoplANNET':<15} {ap_bal_e:>15.4f} {ap_imb_e:>15.4f}")

print("="*80)